In [15]:
import random
import math
def evaluate_board(board):
    return sum(board.values())


def generate_moves(board, depth):
    moves = []
    pieces = [(pos, val) for pos, val in board.items() if val > 0]
    for pos, val in pieces:
        row, col = pos
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1),
                       (-1, -1), (-1, 1), (1, -1), (1, 1)]:
            new_pos = (row + dr, col + dc)
            nr, nc = new_pos
            if 0 <= nr < 8 and 0 <= nc < 8:
                new_board = dict(board)
                del new_board[pos]
                captured = new_board.pop(new_pos, 0)
                new_board[new_pos] = val
                score = evaluate_board(new_board)
                move_desc = f"{pos}->{new_pos}(cap={captured})"
                moves.append((move_desc, new_board, score))
    return moves


def beam_search_chess(board, beam_width=3, depth_limit=3):
    beam = [([], board, evaluate_board(board))]
    best_sequence = None
    best_score = float('-inf')

    for depth in range(depth_limit):
        candidates = []
        for seq, cur_board, cur_score in beam:
            moves = generate_moves(cur_board, depth)
            if not moves:
                continue
            for move_desc, new_board, score in moves:
                new_seq = seq + [move_desc]
                candidates.append((new_seq, new_board, score))

        if not candidates:
            break

        
        candidates.sort(key=lambda x: x[2], reverse=True)
        beam = candidates[:beam_width]

        if beam[0][2] > best_score:
            best_score = beam[0][2]
            best_sequence = beam[0][0]

    return best_sequence, best_score

print("TASK 01 – Beam Search: Chess Move Prediction")

initial_board = {
    (0, 0): 5,  (0, 7): 5,   
    (0, 3): 9,               
    (7, 0): -5, (7, 7): -5,  
    (7, 3): -9,              
    (3, 3): 3,               
    (4, 4): -3,              
}

best_seq, best_score = beam_search_chess(initial_board, beam_width=3, depth_limit=3)
print(f"Best Move Sequence : {best_seq}")
print(f"Evaluation Score   : {best_score}")
print()

TASK 01 – Beam Search: Chess Move Prediction
Best Move Sequence : ['(3, 3)->(4, 4)(cap=-3)']
Evaluation Score   : 3



In [4]:
import math
import random

def calculate_distance(point1, point2):
    return math.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)

def total_route_distance(route, points):
    total = 0
    for i in range(len(route) - 1):
        total += calculate_distance(points[route[i]], points[route[i + 1]])
    total += calculate_distance(points[route[-1]], points[route[0]])
    return total

def get_neighbor(route):
    neighbor = route.copy()
    i, j = random.sample(range(len(route)), 2)
    neighbor[i], neighbor[j] = neighbor[j], neighbor[i]
    return neighbor

def hill_climbing_route(points, max_iterations=1000):
    num_points = len(points)
    current_route = list(range(num_points))
    random.shuffle(current_route)
    current_distance = total_route_distance(current_route, points)
    
    for iteration in range(max_iterations):
        neighbor = get_neighbor(current_route)
        neighbor_distance = total_route_distance(neighbor, points)
        
        if neighbor_distance < current_distance:
            current_route = neighbor
            current_distance = neighbor_distance
    
    return current_route, current_distance

def task02():
    delivery_points = [(0, 0), (2, 3), (5, 1), (6, 4), (3, 5), (1, 2)]
    
    optimized_route, total_distance = hill_climbing_route(delivery_points)
    
    print(f"Optimized route: {optimized_route}")
    print(f"Total distance covered: {total_distance:.2f}")

task02()

Optimized route: [2, 3, 4, 1, 5, 0]
Total distance covered: 17.31


In [5]:
import random
import math

def calculate_distance(city1, city2):
    return math.sqrt((city1[0] - city2[0])**2 + (city1[1] - city2[1])**2)

def total_distance(route, cities):
    distance = 0
    for i in range(len(route) - 1):
        distance += calculate_distance(cities[route[i]], cities[route[i + 1]])
    distance += calculate_distance(cities[route[-1]], cities[route[0]])
    return distance

def fitness(route, cities):
    return 1 / total_distance(route, cities)

def create_random_route(num_cities):
    route = list(range(num_cities))
    random.shuffle(route)
    return route

def select_parents(population, fitness_scores):
    sorted_population = [route for _, route in sorted(zip(fitness_scores, population), reverse=True)]
    return sorted_population[:len(population) // 2]

def crossover(parent1, parent2):
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))
    
    child = [-1] * size
    child[start:end] = parent1[start:end]
    
    pos = end
    for city in parent2[end:] + parent2[:end]:
        if city not in child:
            child[pos % size] = city
            pos += 1
    
    return child

def mutate(route, mutation_rate=0.1):
    if random.random() < mutation_rate:
        i, j = random.sample(range(len(route)), 2)
        route[i], route[j] = route[j], route[i]
    return route

def genetic_algorithm_tsp(cities, population_size=100, generations=500):
    num_cities = len(cities)
    population = [create_random_route(num_cities) for _ in range(population_size)]
    best_route = None
    best_fitness = 0
    
    for generation in range(generations):
        fitness_scores = [fitness(route, cities) for route in population]
        
        if max(fitness_scores) > best_fitness:
            best_fitness = max(fitness_scores)
            best_route = population[fitness_scores.index(best_fitness)]
        
        parents = select_parents(population, fitness_scores)
        
        new_population = []
        while len(new_population) < population_size:
            parent1, parent2 = random.sample(parents, 2)
            child = crossover(parent1, parent2)
            child = mutate(child)
            new_population.append(child)
        
        population = new_population
    
    return best_route, 1/best_fitness

def task03():
    cities = [(random.randint(0, 100), random.randint(0, 100)) for _ in range(10)]
    
    best_route, best_distance = genetic_algorithm_tsp(cities)
    
    print("Cities:", cities)
    print(f"Best route: {best_route}")
    print(f"Total distance: {best_distance:.2f}")

task03()

Cities: [(35, 30), (44, 65), (71, 92), (81, 99), (70, 81), (11, 60), (30, 39), (95, 28), (6, 47), (44, 78)]
Best route: [2, 3, 4, 7, 0, 6, 8, 5, 1, 9]
Total distance: 278.25


In [ ]:
import heapq

def beam_search_task_allocation(jobs, processors, beam_width=3):
    num_processors = len(processors)
    
    def calculate_load(allocation):
        processor_loads = [0] * num_processors
        for job_idx, processor_idx in enumerate(allocation):
            if processor_idx != -1:
                processor_loads[processor_idx] += jobs[job_idx][0]
        return max(processor_loads)
    
    def calculate_priority_score(allocation):
        score = 0
        for job_idx, processor_idx in enumerate(allocation):
            if processor_idx != -1:
                score += jobs[job_idx][1]
        return score
    
    def heuristic(allocation, job_index):
        if job_index >= len(jobs):
            return -calculate_load(allocation)
        
        load = calculate_load(allocation)
        return -load
    
    beam = [(0, [], [-1] * len(jobs))]
    
    for job_idx in range(len(jobs)):
        candidates = []
        
        for score, path, allocation in beam:
            for processor_idx in range(num_processors):
                new_allocation = allocation.copy()
                new_allocation[job_idx] = processor_idx
                new_path = path + [f"Job{job_idx}→P{processor_idx}"]
                new_score = heuristic(new_allocation, job_idx + 1)
                candidates.append((new_score, new_path, new_allocation))
        
        beam = heapq.nlargest(beam_width, candidates, key=lambda x: x[0])
    
    best_score, best_path, best_allocation = beam[0]
    max_load = calculate_load(best_allocation)
    priority_score = calculate_priority_score(best_allocation)
    
    return best_path, best_allocation, max_load, priority_score

def task04():
    jobs = [(5, 3), (3, 5), (8, 2), (2, 4), (6, 1)]
    processors = ['P1', 'P2', 'P3']
    
    best_path, best_allocation, max_load, priority_score = beam_search_task_allocation(jobs, processors)
    
    print("Job allocations:")
    for i, alloc in enumerate(best_allocation):
        print(f"  Job {i} (time={jobs[i][0]}, priority={jobs[i][1]}) → Processor {processors[alloc]}")
    
    print(f"\nMax processor load: {max_load}")
    print(f"Total priority score: {priority_score}")

task04()

Job allocations:
  Job 0 (time=5, priority=3) → Processor P1
  Job 1 (time=3, priority=5) → Processor P2
  Job 2 (time=8, priority=2) → Processor P3
  Job 3 (time=2, priority=4) → Processor P1
  Job 4 (time=6, priority=1) → Processor P2

Max processor load: 9
Total priority score: 15
